In [ ]:
from wotan import flatten
import matplotlib.pyplot as plt
import pandas as pd
from astropy.timeseries import LombScargle
import numpy as np
from astropy.timeseries import BoxLeastSquares

In [ ]:
# cargar datos
file = pd.read_csv("tic_427395136_s_6_corrected.csv")

#print(file.columns)
time = file["time"]
flux = file["flux"]
#print(time,flux)

# plot 
plt.scatter(time, flux, s=5)

In [ ]:
def explore_detrend(window_length, break_tol):

    flat, trend = flatten(
        time,
        flux,
        window_length=window_length,
        break_tolerance=break_tol,
        method="hspline",
        return_trend=True
    )

    plt.figure(figsize=(12,5))
    plt.scatter(time, flux, s=3, color='k', label="Original")
    plt.plot(time, trend, color='r', label="Trend")
    plt.legend()
    plt.title(f"window={window_length}, break={break_tol}")
    plt.show()

    return flat, trend

In [ ]:
flat_flux, trend_flux = explore_detrend(.85, 1.0)

df = pd.DataFrame({
    "time": time,          
    "flat_flux": flat_flux,
    "trend_flux": trend_flux
})
t0_B = time[np.argmin(flat_flux)]
print(t0_B)
df.to_csv("TIC_199780530_s_6_trending.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
from astropy.timeseries import LombScargle

# =========================
# 1. Cargar datos
# =========================
file = pd.read_csv("tic_427395136_s_6_corrected.csv")

time = file["time"].values
flux = file["flux"].values

# =========================
# 2. Limpieza básica
# =========================
mask = np.isfinite(time) & np.isfinite(flux)
time = time[mask]
flux = flux[mask]

# =========================
# 3. LightCurve
# =========================
lc = lk.LightCurve(time=time, flux=flux)

# =========================
# 4. Detrending
# =========================
lc_flat = lc.flatten(window_length=201)
lc_flat = lc_flat.remove_outliers(sigma=5)

# =========================
# 5. BLS → PERIODO ORBITAL
# =========================
bls = lc_flat.to_periodogram(
    method='bls',
    minimum_period=0.5,
    maximum_period=10,
    frequency_factor=5
)

P_orb = bls.period_at_max_power.value
t0 = bls.transit_time_at_max_power.value

print(f"Periodo orbital (BLS): {P_orb:.5f} días")
print(f"t0: {t0:.5f}")

# =========================
# 6. PROFUNDIDAD (ROBUSTA)
# =========================
mask_transit = bls.get_transit_mask(
    period=P_orb,
    transit_time=bls.transit_time_at_max_power,
    duration=bls.duration_at_max_power
)

flux_in = lc_flat.flux.value[mask_transit]
flux_out = lc_flat.flux.value[~mask_transit]

F_in = np.median(flux_in)
F_out = np.median(flux_out)

depth = 1 - (F_in / F_out)

print(f"Profundidad (global): {depth:.6f}")
print(f"Rp/Rs ≈ {np.sqrt(depth):.4f}")

# =========================
# 7. FASEADO (P y 2P)
# =========================
lc_fold = lc_flat.fold(period=P_orb, epoch_time=t0)
lc_fold_2 = lc_flat.fold(period=2*P_orb, epoch_time=t0)

# =========================
# 8. DETECTAR PRIMARIO / SECUNDARIO
# =========================
phase = lc_fold_2.phase.value
flux_phase = lc_fold_2.flux.value

# baseline global
F_out_phase = np.median(flux_phase)

# ventanas aproximadas
mask_primary = (phase > -0.1) & (phase < 0.1)
mask_secondary = (phase > 0.4) & (phase < 0.6)

F_primary = np.median(flux_phase[mask_primary])
F_secondary = np.median(flux_phase[mask_secondary])

depth_primary = 1 - (F_primary / F_out_phase)
depth_secondary = 1 - (F_secondary / F_out_phase)

print("\n--- Binaria eclipsante ---")
print(f"Profundidad primaria:   {depth_primary:.6f}")
print(f"Profundidad secundaria: {depth_secondary:.6f}")

# =========================
# 9. PLOTS
# =========================

# --- BLS ---
bls.plot()
plt.title("BLS Periodogram")
plt.show()

# --- Fase P ---
lc_fold.scatter()
plt.title("Fase con P")
plt.show()

# --- Fase 2P ---
plt.figure()
plt.scatter(phase, flux_phase, s=3)

plt.axhline(F_out_phase, color='green', label='Baseline')
plt.axhline(F_primary, color='red', label='Primario')
plt.axhline(F_secondary, color='orange', label='Secundario')

plt.legend()
plt.title("Fase con 2P (binaria eclipsante)")
plt.xlabel("Fase")
plt.ylabel("Flux")
plt.show()

# =========================
# 10. REMOVER ECLIPSES
# =========================
lc_no_transit = lc_flat[~mask_transit]

t = lc_no_transit.time.value
f = lc_no_transit.flux.value

mask = np.isfinite(t) & np.isfinite(f)
t = t[mask]
f = f[mask]

# =========================
# 11. LOMB-SCARGLE (ROTACIÓN)
# =========================
min_period = 0.2
max_period = 20

frequency, power = LombScargle(t, f).autopower(
    minimum_frequency=1/max_period,
    maximum_frequency=1/min_period
)

best_frequency = frequency[np.argmax(power)]
P_rot = 1 / best_frequency

print(f"\nPeriodo de rotación: {P_rot:.5f} días")

# =========================
# 12. PLOT LS
# =========================
plt.figure()
plt.plot(1/frequency, power)
plt.xlabel("Periodo (días)")
plt.ylabel("Power")427393298
plt.title("Lomb-Scargle")
plt.gca().invert_xaxis()
plt.show()

# =========================
# 13. FASE ROTACIONAL
# =========================
phase_rot = (t % P_rot) / P_rot

plt.figure()
plt.scatter(phase_rot, f, s=2)
plt.xlabel("Fase")
plt.ylabel("Flux")
plt.title("Modulación rotacional")
plt.show()